# Problem 032:  Pandigital Products

> We shall say that an $n$-digit number is pandigital if it makes use of all the digits $1$ to $n$ exactly once; for example, the $5$-digit number, $15234$, is $1$ through $5$ pandigital.
>
> The product $7254$ is unusual, as the identity, $39 \times 186 = 7254$, containing multiplicand, multiplier, and product is $1$ through $9$ pandigital.
>
> Find the sum of all products whose multiplicand/multiplier/product identity can be written as a $1$ through $9$ pandigital.
>
> HINT: Some products can be obtained in more than one way so be sure to only include it once in your sum.

## Naive solution $\mathcal O(N\ln N 10^{\lfloor (N+1) / 2 \rfloor})$

A quick and dirty way is to look at all products that will give the right number of digits in the entire equation and check if each equation is pandigital or not.  To make the problem a little broader, the algorithm will work for determining if the product equation is a palindrome of any set of numbers, not just pandigital.

Consider the product of an $n$-digit and an $m$-digit number.  Their product will either be $n+m-1$ or $n+m$ digits long, which can be seen by considering the minimum product possible, $10^{n-1} \times 10^{m-1} = 10^{n+m-2}$ and maximum product possible, $(10^n - 1) \times (10^m - 1) = 10^{n+m} - 10^n - 10^m + 1$.  Thus, the concatentated string of all three values will be of length $2(n+m)+\{0, 1\}$.  When searching for a palindrome of length $N$, the two numbers being multiplied will have a total length of $n+m = \lfloor (N+1) / 2 \rfloor$, and thus the smaller number must $\lfloor (N+1) / 4 \rfloor$, or less.  This gives the bound of the search necessary to exhaust all possibilities.

Next, a quick check for whether the equation is a palindrome.  The target list of digits is first codified into a string with the digits in increasing order.  Then to check if a number is a palidrome, it is a matter of converting the three values into strings, concatenating them, and sorting the characters.  If the given equation produces the same string as the target string, they are palindromes.

To get rid of any duplicates that come up in the search, the list of all hits is converted to a set, as per the hint in the problem.

The search space has a size of $\sim 10^{\lfloor (N+1) / 2 \rfloor}$ and each check requires a sort of $N$ characters, thus the overall time complexity is $\mathcal O(N\ln N 10^{\lfloor (N+1) / 2 \rfloor})$.

In [112]:
%%time

def is_pandigital_product(i: int, j: int, n_str: str):
    return n_str == ''.join(sorted(str(i) + str(j) + str(i*j)))

def p032_naive(N: int) -> int:
    n_str = "".join(map(str,range(1,N+1)))

    s = set()
    for ipow in range((N + 1) // 4):
        imin = pow(10, ipow)
        imax = 10 * imin
        jmin = pow(10, (N + 1) // 2 - ipow - 2)
        jmax = 10 * jmin
        s |= set([i * j for i in range(imin, imax) for j in range(jmin, jmax) if is_pandigital_product(i, j, n_str)])
        
    return sum(s)

N = 9
ans = p032_naive(N)

print(ans)

45228
CPU times: user 432 ms, sys: 0 ns, total: 432 ms
Wall time: 445 ms


## Dynamic Programming solution $\mathcal O(N10^{\lfloor (N+1) / 2 \rfloor})$

A lot of unnecessary computations are being done in the above algorithm.  Some of it could be mitigated by setting stricter limits on the search to make sure only the correct length strings are compared.  But there are other aspects like searching for all the pairs of numbers to be multiplied that share values themselves.  I find that a dynamic programming approach is much better suited to avoid unnnecessary computations of this type.

The search space of the smallest number to be multiplied is the same as above, but we can first weed out any values that will already make it impossible to get the desired target string, such as multiples of $10$ and repeating digits like multiples of $11$ in the presented case.  Then the multiplier is "grown" from the one's position, calculating the corresponding digit in the product as well.  In this way, the search can be pruned whenever a combination of digits will make it impossible to get the desired palindrome.

The code is a lot more complicated, and I'm not convinced that the time complexity is even lowered at all from running a few time tests, but the timing is much better than the first attempt above.  For example, it takes about a muinute to run a palindrome of length $13$ for the first code, whereas I can get a length of $18$ in that time with the code below.

In [110]:
%%time

def p032(N: int) -> int:
    s = set()

    digits = [0] * 10
    for i in range(N):
        digits[i % 9 + 1] += 1
    
    carry = [None] * N
    mul = [None] * N
    prd = [None] * N
    

    imax = pow(10, (N + 1) // 4)
    
    for i in range(imax):
        # check if `i` itself is a valid number
        igo = False
        for d in map(int, str(i)):
            if not digits[d]:
                igo = True
            digits[d] -= 1
        if igo:
            for d in map(int, str(i)):
                digits[d] += 1
            continue


        # Start DP
        nd = len(str(i))
        n = 1
        
        carry[0] = 0
        mul[0] = 0
        while n != 0:
            # check that the newest digit in the multiplier is available
            m = mul[n-1]
            if not digits[m]:
                m = mul[n-1] + 1
                while n != 0 and m == 10:
                    n -= 1
                    if n:
                        m = mul[n-1] + 1
                        digits[m-1] += 1
                        p = prd[n-1]
                        digits[p] += 1
                        nd -= 2
                if n:
                    mul[n-1] = m
                continue
            
            digits[m] -= 1
            nd += 1            
            
            # check that the newest digit in the product is available
            p = carry[n-1] + i * m
            c = p // 10
            p %= 10
            prd[n-1] = p
            carry[n] = c
            if not digits[p]:
                digits[m] += 1
                nd -= 1
                m = mul[n-1] + 1
                while n != 0 and m == 10:
                    n -= 1
                    if n:
                        m = mul[n-1] + 1
                        digits[m-1] += 1
                        p = prd[n-1]
                        digits[p] += 1
                        nd -= 2
                if n:
                    mul[n-1] = m
                continue

            digits[p] -= 1
            nd += 1

            # check if the current step is a complete palindrome
            if (nd == N and c == 0) or (c < 10 and c != 0 and nd == N - 1 and digits[c] == 1):
                k = ''.join(map(str,prd[n-1::-1]))
                if nd != N:
                    k = str(c) + k
                s |= set([int(k)])
                digits[m] += 1
                digits[p] += 1
                nd -= 2
                m = mul[n-1] + 1
                while n != 0 and m == 10:
                    n -= 1
                    if n:
                        m = mul[n-1] + 1
                        digits[m-1] += 1
                        p = prd[n-1]
                        digits[p] += 1
                        nd -= 2
                if n:
                    mul[n-1] = m
                continue
            
            # if all the checks above pass, then the multiplier is extended by a digit
            mul[n] = 0
            n += 1

        for d in map(int, str(i)):
            digits[d] += 1
    
    return sum(s)

N = 9
ans = p032(N)

print(ans)

45228
CPU times: user 6.15 ms, sys: 1e+03 ns, total: 6.16 ms
Wall time: 5.99 ms
